In [1]:
# Install dependencies
%pip install ultralytics
%pip install opencv-python
%pip install simple_pid
%pip install numpy
%pip install pyserial
%pip install cv2-enumerate-cameras
%pip install scikit-learn
%pip install nolds
%pip install folium
%pip install html2image

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [ ]:
# Configurations
import cv2
import math
import serial
import time
import nolds
import requests
import folium
import webbrowser
import os
import numpy as np
import matplotlib.pyplot as plt
from ultralytics import YOLO
from simple_pid import PID
from cv2_enumerate_cameras import enumerate_cameras
from collections import deque
from sklearn.decomposition import PCA
from html2image import Html2Image

# Camera configuration
CAMERA_INDEX_1 = 1200                  # Index of the camera  to be used (custom value; check with enumerate_cameras)
CAMERA_INDEX_2 = 1201                  # Index of the camera to be used (custom value; check with enumerate_cameras)

# Movement vector filtering
VECTOR_MIN_LENGTH = 20                 # Minimum length of movement vector to consider (in pixels)
VECTOR_MAX_LENGTH = 200                # Maximum length of movement vector to consider (in pixels)
PIXELS_PER_UNIT = 20000.0              # Conversion factor from pixels to real-world units (e.g., cm)
VECTOR_HISTORY_SIZE = 10               # Number of previous positions to store for movement vector calculation

# Geometry setup
CAMERA_DISTANCE = 0.43                 # Distance between two cameras (used for triangulation)
DISTANCE_UNIT = "m"                    # Unit of measurement for camera distance
PROTOTYPE_ANGLE = 10                   # Angle of the prototype setup (in degrees)
R = 6371000                            # Radius of the Earth in DISTANCE_UNITs


# YOLO model setup
MODEL = YOLO('yolov10m.pt')            # Load YOLOv8 nano model for object detection
CLASS_NAMES = MODEL.names              # List of all class labels in the model
DESIRED_CLASSES = ["sports ball"]      # Classes we want to detect (only track sports balls)

# PID control parameters
PID_KP = 0.015                         # PID Proportional gain
PID_KI = 0.003                         # PID Integral gain
PID_KD = 0.09                          # PID Derivative gain
PID_OUTPUT_LIMITS = (-3, 3)            # Output limits for PID correction to smooth servo movement
ALPHA = 0.2                            # Smoothing factor for servo movement (0 < ALPHA < 1)

# Servo configuration
SERVO_MIN = 0                          # Minimum angle a servo can rotate to
SERVO_MAX = 180                        # Maximum angle a servo can rotate to
SERVO_MIN_STEP = 1                     # Minimum step size in angles for servo movement
SERVO_SCAN_STEP = 5                    # Step size in angles for servo scanning movements
SERVO_DOUBLE_SCAN_INTERVAL = 300       # Seconds between servo scan movements when no cameras found an object
SERVO_SINGLE_SCAN_INTERVAL = 10        # Seconds between servo scan movements when only one camera found the object
DELAY_BETWEEN_SERVO_MOVEMENTS = 1      # Delay in seconds between servo movements to allow physical movement
SERVO_INIT = {                         # Initial servo positions (centered)
    'x1': 65, 
    'y1': 90, 
    'x2': 115, 
    'y2': 90
}

# Arduino serial communication
ARDUINO_PORT = '/dev/cu.usbmodem11301'  # Serial port where Arduino is connected
ARDUINO_BAUDRATE = 9600                 # Communication speed between Python and Arduino

# Weather API configuration
LAT_INIT = -23.451642                         # Latitude of camera 1
LON_INIT = -46.774476                         # Longitude of camera 1
API_KEY = '860d2355fa0fe182dce220585a819603'  # OpenWeatherMap API key
PREDICTION_INTERVAL = 30                      # Seconds between flight predictions
FIRST_PREDICTION_INTERVAL = 10                # Seconds until first flight prediction

# Telegram bot configuration
TOKEN = "8750939695:AAHakcI-1T_IC6WkK2CvcPSsSsAT5hnzCrI" # Change to your bot token
CHAT_ID = "-4856182101"                  # Change to your chat ID
ALERT_COOLDOWN = 30                     # Seconds between alerts

In [3]:
# CV2 and camera settings

# Initialize camera
def init_camera(index, verbose=True):
    cap = cv2.VideoCapture(index)
    if not cap.isOpened():
        raise RuntimeError(f"Error: Could not open webcam {index}.")
    if verbose:
        print(f"Camera {index} opened successfully.")
    return cap


# Read current frame
def read_frame(cap):
    ret, frame = cap.read()
    if not ret:
        raise RuntimeError("Failed to capture image from webcam.")
    return frame


# Calculate frame center
def calculate_center(frame):
    height, width = frame.shape[:2]
    x_center, y_center = width / 2, height / 2
    return x_center, y_center


# Check if class is in desired classes
def valid_class(box, class_names=CLASS_NAMES, desired_classes=DESIRED_CLASSES):
    return class_names[int(box.cls)] in desired_classes


# Draw bounding box
def draw_bounding_box(frame, box, class_names=CLASS_NAMES, verbose=True):
    x1, y1, x2, y2 = map(int, box.xyxy[0])
    cv2.rectangle(frame, (x1, y1), (x2, y2), (255, 0, 255), 3)
    
    cls = int(box.cls[0])
    confidence = math.ceil((box.conf[0] * 100)) / 100
    
    org = [x1, y1]
    font = cv2.FONT_HERSHEY_SIMPLEX
    font_scale = 1
    color = (255, 0, 0)
    thickness = 2
    cv2.putText(frame, class_names[cls], org, font, font_scale, color, thickness)
    
    if verbose:
        print(f"Class: {class_names[cls]}, Confidence: {confidence}")


# Draw movement vector
def draw_movement_vector(frame, box, balloon_position, vector_history, avg_dt, prev_position=None, verbose=True):
    x1, y1, x2, y2 = map(int, box.xyxy[0])
    cx_pixel, cy_pixel = int((x1 + x2) / 2), int((y1 + y2) / 2)
    x, y, z = float(balloon_position[0]), float(balloon_position[1]), float(balloon_position[2])

    if len(vector_history) == 0:
        base_px, base_py, base_pz = prev_position if prev_position is not None else (x, y, z)
    else:
        base_px = sum(p[0] for p in vector_history) / len(vector_history)
        base_py = sum(p[1] for p in vector_history) / len(vector_history)
        base_pz = sum(p[2] for p in vector_history) / len(vector_history)

    dx_w, dy_w, dz_w = x - base_px, y - base_py, z - base_pz
    mag_w = math.sqrt(dx_w**2 + dy_w**2 + dz_w**2)
    vel_w = mag_w / avg_dt

    if mag_w > 1e-9:
        ux, uy = dx_w / mag_w, dz_w / mag_w
        length_px = mag_w * PIXELS_PER_UNIT
        length_px = max(VECTOR_MIN_LENGTH, min(length_px, VECTOR_MAX_LENGTH))
        
        end_x = int(cx_pixel + ux * length_px)
        end_y = int(cy_pixel - uy * length_px)
        cv2.arrowedLine(frame, (cx_pixel, cy_pixel), (end_x, end_y), (0, 0, 255), 3, tipLength=0.3)

    vector_history.append((x, y, z))
    new_prev_position = (x, y, z)

    if verbose:
        print(f"3D displacement: ({dx_w:.3f}, {dy_w:.3f}, {dz_w:.3f})")
        print(f"Velocity: {vel_w:.3f} {DISTANCE_UNIT}/s")

    return new_prev_position

In [4]:
# Calculations

# Calculate balloon coordinates
def triangulation(angles: dict, d=CAMERA_DISTANCE, verbose=True):
    alpha = np.radians(angles['x1'])
    beta = np.radians(angles['x2']-90)
    gamma = np.radians(90-angles['y1'])
    delta = np.radians(90-angles['y2'])
    
    if (np.sin(alpha + beta)) == 0:
        raise ValueError("Invalid angles for triangulation.")
    
    n = d * np.sin(beta) / np.sin(alpha + beta)
    m = d * np.sin(alpha) / np.sin(alpha + beta)

    x1 = n * np.cos(alpha)
    y1 = n * np.sin(alpha)
    z1 = n * np.tan(gamma)
    
    x2 = d - (m * np.cos(beta))
    y2 = m * np.sin(beta)
    z2 = m * np.tan(delta)

    x = (x1 + x2) / 2
    y = (y1 + y2) / 2
    z = (z1 + z2) / 2
    
    if verbose:
        print(f"Balloon position: ({x:.2f}, {y:.2f}, {z:.2f})")
    
    return np.array([x, y, z])


# Calculate distance to camera
def get_distance(camera, balloon_position, d=CAMERA_DISTANCE, verbose=True):
    if camera == 'C1':
        camera_position = np.array([0, 0, 0])
    elif camera == 'C2':
        camera_position = np.array([d, 0, 0])
    else:
        raise ValueError("Invalid camera ID. Use 'C1' or 'C2'.")
    
    distance = np.linalg.norm(balloon_position - camera_position)
    
    if verbose:
        print(f"{camera} Distance: {distance:.2f} {DISTANCE_UNIT}")
    
    return distance


# Compute Lyapunov exponent
def compute_lyapunov_exponent(positions, dt, method="pca", emb_dim=6, tau=1, min_tsep=10, trajectory_len=20, debug_plot=True):
    positions = np.array(positions)
    if positions.shape[1] != 3:
        raise ValueError("Positions must be of shape (N,3).")

    if method.lower() == "pca":
        pca = PCA(n_components=1)
        scalar_series = pca.fit_transform(positions).flatten()
    elif method.lower() == "norm":
        scalar_series = np.linalg.norm(positions, axis=1)
    elif method.lower() in ["x", "y", "z"]:
        idx = {"x":0, "y":1, "z":2}[method.lower()]
        scalar_series = positions[:, idx]
    else:
        raise ValueError("Invalid method. Choose 'pca', 'norm', 'x', 'y', or 'z'.")

    scalar_series = (scalar_series - np.mean(scalar_series)) / (np.std(scalar_series) + 1e-12)

    lyap_raw = nolds.lyap_r(
        scalar_series,
        emb_dim=emb_dim,
        tau=tau,
        min_tsep=min_tsep,
        trajectory_len=trajectory_len,
        debug_plot=debug_plot
    )
    lyap_exp = lyap_raw / dt
    lyap_time = 1 / lyap_exp if lyap_exp > 0 else float('inf')

    return lyap_exp, lyap_time

In [5]:
# PID controller

# Initialize PID controller
def init_pid(x_center, y_center, verbose=True):
    global prev_pan, prev_tilt
    prev_pan, prev_tilt = 0, 0
    pid_pan = PID(Kp=PID_KP, Ki=PID_KI, Kd=PID_KD, setpoint=x_center)
    pid_pan.output_limits = PID_OUTPUT_LIMITS
    pid_tilt = PID(Kp=PID_KP, Ki=PID_KI, Kd=PID_KD, setpoint=y_center)
    pid_tilt.output_limits = PID_OUTPUT_LIMITS
    if verbose:
        print("PID controller initialized.")
    return pid_pan, pid_tilt


# Calculate corrections using PID controller
def calculate_corrections(box, pid_pan, pid_tilt, verbose=True):
    global prev_pan, prev_tilt
    x1, y1, x2, y2 = box.xyxy[0]
    x_box = float((x1 + x2) / 2)
    y_box = float((y1 + y2) / 2)
    
    pan_raw = pid_pan(x_box)
    tilt_raw = pid_tilt(y_box)
    pan = ALPHA * pan_raw + (1 - ALPHA) * prev_pan
    tilt = ALPHA * tilt_raw + (1 - ALPHA) * prev_tilt
    prev_pan, prev_tilt = pan, tilt
    
    if verbose:
        print(f"Box: ({x_box}, {y_box})")
        print(f"Pan: {pan}, Tilt: {tilt}")
    return pan, tilt


# Update servo angle
def update_angle(servo_angle, correction):
    if correction > 10:
        correction = 10
    elif correction < -10:
        correction = -10
    elif abs(correction) < SERVO_MIN_STEP:
        correction = 0
    return max(SERVO_MIN, min(SERVO_MAX, math.ceil(servo_angle + correction)))

In [6]:
# Arduino

# Initialize serial connection to arduino
def init_arduino(*, port=ARDUINO_PORT, baudrate=ARDUINO_BAUDRATE, initial_angles=SERVO_INIT, verbose=True):
    try:
        arduino = serial.Serial(port, baudrate)
        time.sleep(2)
        if verbose:
            print(f"Connected to Arduino on {port} at {baudrate} baud.")
        move_servos(arduino, initial_angles)
        return arduino
    except serial.SerialException as e:
        raise RuntimeError(f"Could not connect to Arduino on {port}: {e}")


# Move servos
def move_servos(arduino, angles: dict, verbose=True):
    try:
        command = f"{angles['x1']},{angles['y1']},{angles['x2']},{angles['y2']}\n"
        arduino.write(command.encode())
        arduino.flush()
        if verbose:
            print(f"C1 → X: {angles['x1']:.1f}°, Y: {angles['y1']:.1f}°")
            print(f"C2 → X: {angles['x2']:.1f}°, Y: {angles['y2']:.1f}°")
    except serial.SerialException as e:
        print(f"Serial communication error: {e}")


# Rotate servos to scan environment
def rotate_servos(arduino, angles: dict, pos, c1=True, c2=True, step=SERVO_SCAN_STEP, delay=DELAY_BETWEEN_SERVO_MOVEMENTS, verbose=True):
    try:
        if verbose:
            print(f"Scanning environment...")
        if c1:
            angles['x1'] = (step * pos) % SERVO_MAX
            if (step * pos) > (2 * SERVO_MAX):
                angles['y1'] = 180
            elif (5 * pos) > SERVO_MAX:
                angles['y1'] = 135
            else:
                angles['y1'] = 90
            if verbose:
                print(f"Rotating C1 to x={angles['x1']}, y={angles['y1']}")
        if c2:
            angles['x2'] = (step * pos) % SERVO_MAX
            if (step * pos) > (2 * SERVO_MAX):
                angles['y2'] = 180
            elif (5 * pos) > SERVO_MAX:
                angles['y2'] = 135
            else:
                angles['y2'] = 90
            if verbose:
                print(f"Rotating C2 to x={angles['x2']}, y={angles['y2']}")
        command = f"{angles['x1']},{angles['y1']},{angles['x2']},{angles['y2']}\n"
        arduino.write(command.encode())
        arduino.flush()
        time.sleep(DELAY_BETWEEN_SERVO_MOVEMENTS)
    except serial.SerialException as e:
        print(f"Serial communication error: {e}")

In [7]:
# Flight prediction

# Calculate latitude and longitude
def calculate_lat_lon(x, y, lat0=LAT_INIT, lon0=LON_INIT, theta_deg=0):
    theta_rad = np.radians(theta_deg)
    
    x_geo = x * np.cos(theta_rad) - y * np.sin(theta_rad)
    y_geo = x * np.sin(theta_rad) + y * np.cos(theta_rad)
    
    delta_lat = (y_geo / R) * (180 / np.pi)
    
    lat0_rad = np.radians(lat0)
    delta_lon = (x_geo / (R * np.cos(lat0_rad))) * (180 / np.pi)
    
    lat1 = lat0 + delta_lat
    lon1 = lon0 + delta_lon
    
    return lat1, lon1


# Convert latitude and longitude to x, y coordinates
def lat_lon_to_xy(lat1, lon1, lat0=LAT_INIT, lon0=LON_INIT, theta_degrees=0):
    
    theta_rad = np.radians(theta_degrees)
    
    y_aligned = (lat1 - lat0) * (np.pi / 180) * R
    lat0_rad = np.radians(lat0)
    x_aligned = (lon1 - lon0) * (np.pi / 180) * R * np.cos(lat0_rad)
    
    x = x_aligned * np.cos(theta_rad) + y_aligned * np.sin(theta_rad)
    y = -x_aligned * np.sin(theta_rad) + y_aligned * np.cos(theta_rad)
    
    return x, y


# Get wind data from OpenWeatherMap API
def get_wind_data(lat, lon, api_key=API_KEY, hour=0):
    url = f"https://api.openweathermap.org/data/3.0/onecall?lat={lat}&lon={lon}&appid={api_key}"
    
    try:
        response = requests.get(url)
        response.raise_for_status()
        data = response.json()
        
        if hour == 0:
            wind_speed = data['current']['wind_speed']
            wind_deg = data['current']['wind_deg']
        else:
            wind_speed = data['hourly'][hour - 1]['wind_speed']
            wind_deg = data['hourly'][hour - 1]['wind_deg']
        
        return wind_speed, wind_deg
    
    except (requests.exceptions.RequestException, KeyError, IndexError) as error:
        print(f"Error retrieving wind data: {error}")
        return None, None


# Move balloon one hour based on wind data
def move_balloon_one_hour(lat0, lon0, wind_speed, wind_deg):        
    displacement_m = wind_speed * 3600
    
    wind_rad = np.radians(wind_deg)
    
    displacement_ew = displacement_m * np.sin(wind_rad)
    displacement_ns = displacement_m * np.cos(wind_rad)
    
    delta_lat = (displacement_ns / R) * (180 / np.pi)
    
    lat0_rad = np.radians(lat0)
    delta_lon = (displacement_ew / (R * np.cos(lat0_rad))) * (180 / np.pi)
    
    lat_final = lat0 + delta_lat
    lon_final = lon0 + delta_lon
    
    return lat_final, lon_final


# Predict balloon flight over 48 hours
def predict_balloon_flight(lat_init, lon_init, api_key=API_KEY, verbose=True):    
    positions = []
    lat_current = lat_init
    lon_current = lon_init
    if verbose:
        print(f"Starting prediction at lat: {lat_current}, lon: {lon_current}")
    
    for hour in range(49):
        wind_speed, wind_deg = get_wind_data(lat_current, lon_current, api_key, hour)
        
        if verbose and (wind_speed is None or wind_deg is None):
            print(f"Skipping hour {hour} due to missing wind data.")
            continue
        
        lat_current, lon_current = move_balloon_one_hour(lat_current, lon_current, wind_speed, wind_deg)
        positions.append((lat_current, lon_current))
        
        if verbose:
            print(f"Hour {hour}: {positions[hour]} --> (Speed = {wind_speed} m/s, Deg = {wind_deg}°)")
    return positions


# Plot trajectory on graphs
def plot_graph_trajectory(data):
    lats = [p[0] for p in data]
    lons = [p[1] for p in data]
    hours = list(range(len(data)))
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))
    
    ax1.plot(lons, lats, 'b-', linewidth=2, label='Trajetória do balão')
    ax1.scatter(lons[0], lats[0], color='green', s=200, zorder=5, label='Início (hora 0)', marker='o')
    ax1.scatter(lons[-1], lats[-1], color='red', s=200, zorder=5, label='Fim (hora 48)', marker='X')
    
    for i in range(0, len(data), 6):
        ax1.annotate(f'{i}h', (lons[i], lats[i]), fontsize=8, ha='center')
    
    ax1.set_xlabel('Longitude (°)', fontsize=12)
    ax1.set_ylabel('Latitude (°)', fontsize=12)
    ax1.set_title('Trajetória do Balão - 48 Horas', fontsize=14, fontweight='bold')
    ax1.grid(True, alpha=0.3)
    ax1.legend(fontsize=10)
    ax1.invert_yaxis()
    
    ax2_lat = ax2
    ax2_lon = ax2.twinx()
    
    line1 = ax2_lat.plot(hours, lats, 'b-', linewidth=2, label='Latitude')
    ax2_lat.set_xlabel('Tempo (horas)', fontsize=12)
    ax2_lat.set_ylabel('Latitude (°)', fontsize=12, color='b')
    ax2_lat.tick_params(axis='y', labelcolor='b')
    ax2_lat.grid(True, alpha=0.3)
    
    line2 = ax2_lon.plot(hours, lons, 'r-', linewidth=2, label='Longitude')
    ax2_lon.set_ylabel('Longitude (°)', fontsize=12, color='r')
    ax2_lon.tick_params(axis='y', labelcolor='r')
    
    ax2.set_title('Evolução de Latitude e Longitude', fontsize=14, fontweight='bold')
    
    lines = line1 + line2
    labels = [l.get_label() for l in lines]
    ax2_lat.legend(lines, labels, loc='upper left', fontsize=10)
    
    plt.tight_layout()
    plt.show()


# Plot map trajectory
def plot_map_trajectory(data):
    centro_lat = (data[0][0] + data[-1][0]) / 2
    centro_lon = (data[0][1] + data[-1][1]) / 2
    
    mapa = folium.Map(
        location=[centro_lat, centro_lon],
        zoom_start=8,
        tiles='OpenStreetMap'
    )
    
    folium.PolyLine(
        data,
        color='blue',
        weight=3,
        opacity=0.8,
        popup='Trajetória do balão'
    ).add_to(mapa)
    
    folium.Marker(
        location=data[0],
        popup='<b>Início</b><br>Hora 0<br>São Paulo',
        icon=folium.Icon(color='green', icon='play'),
        tooltip='Início (hora 0)'
    ).add_to(mapa)
    
    folium.Marker(
        location=data[-1],
        popup='<b>Fim</b><br>Hora 48<br>SC/PR',
        icon=folium.Icon(color='red', icon='stop'),
        tooltip='Fim (hora 48)'
    ).add_to(mapa)
    
    for i in range(6, len(data), 6):
        folium.CircleMarker(
            location=data[i],
            radius=5,
            popup=f'<b>Hora {i}</b><br>Lat: {data[i][0]:.4f}<br>Lon: {data[i][1]:.4f}',
            color='blue',
            fill=True,
            fillOpacity=0.7,
            tooltip=f'Hora {i}'
        ).add_to(mapa)
    
    for i in range(0, len(data), 6):
        folium.Marker(
            location=data[i],
            icon=folium.DivIcon(html=f'''
                <div style="font-size: 12pt; color: black; font-weight: bold; 
                            background-color: white; border-radius: 50%; 
                            width: 25px; height: 25px; text-align: center; line-height: 25px;">
                    {i}h
                </div>'''),
            popup=f'Hora {i}'
        ).add_to(mapa)
    
    mapa.save('trajetoria_balao.html')
    webbrowser.open('file://' + os.path.realpath('trajetoria_balao.html'))
    
    hti = Html2Image()
    hti.screenshot(
        html_file='trajetoria_balao.html',
        save_as='trajetoria_balao.png',
        size=(1200, 800)
    )

In [8]:
# Telegram alert

# Check if enough time has passed since last alert
def valid_time(current_time, last_time, interval):
    return (current_time - last_time) >= interval


# Send Telegram alert with balloon coordinates
def send_telegram_alert(position, dist_c1, dist_c2, message="🎈 Balão detectado!"):
    x, y, z = position
    message = f"🎈 Balão detectado!\n" \
            f"📍 Coordenadas: x={x:.2f}, y={y:.2f}, z={z:.2f}\n"
    
    if dist_c1 is not None:
        message += f"📏 Distância C1: {dist_c1:.2f} cm\n"
    if dist_c2 is not None:
        message += f"📏 Distância C2: {dist_c2:.2f} cm\n"

    url = f"https://api.telegram.org/bot{TOKEN}/sendMessage"
    data = {"chat_id": CHAT_ID, "text": message}
    response = requests.post(url, data=data)
    
    if response.status_code == 200:
        print("📩 Coordenadas enviadas com sucesso!")
    else:
        print("⚠️ Erro ao enviar coordenadas:", response.text)
    
    try:
        with open("trajetoria_balao.html", "rb") as f:
            requests.post(
                f"https://api.telegram.org/bot{TOKEN}/sendDocument",
                data={"chat_id": CHAT_ID},
                files={"document": f}
            )
        with open("trajetoria_balao.png", "rb") as f:
            requests.post(
                f"https://api.telegram.org/bot{TOKEN}/sendPhoto",
                data={"chat_id": CHAT_ID},
                files={"photo": f}
            )
    except Exception as e:
        print("⚠️ Erro ao enviar mapa:", e)

In [9]:
# Print functions

# Print divider
def print_div(text):
    print("\n" + f" {text} ".center(100, '-') + "\n")


# Print available cameras
def print_cameras():
    for camera_info in enumerate_cameras():
        print(f'{camera_info.index}: {camera_info.name}')


# Print final information
def final_prints(frame_count, x1_center, y1_center, x2_center, y2_center, frame_delay, balloon_position, dist_c1, dist_c2, servo_angles, lyap_exp, lyap_time):
    x, y, z = balloon_position
    print(f"Total frames: {frame_count}, Total time: {frame_delay*frame_count:.2f} seconds")
    print(f"Detected classes: {DESIRED_CLASSES}")
    print(f"Frame centers: ({x1_center}, {y1_center}), ({x2_center}, {y2_center})")
    print(f"FPS: {(1/frame_delay):.2f}")
    print(f"Final balloon position: ({x:.2f}, {y:.2f}, {z:.2f})")
    print(f"Final C1 distance: {dist_c1:.2f} {DISTANCE_UNIT}")
    print(f"Final C2 distance: {dist_c2:.2f} {DISTANCE_UNIT}")
    print(f"Servo angles: {servo_angles}")
    print(f"Lyapunov exponent: {lyap_exp:.4f} per second")
    if lyap_time != float('inf'):
        print(f"Predictability horizon: {lyap_time:.2f} seconds\n")
    else:
        print("Predictability horizon: Infinite (non-chaotic behavior)\n")

In [ ]:
# Main program
print_div("Available cameras")
print_cameras()

print_div("Initial settings")
frame_count, total_time = 0, 0
last_prediction_time, first_time, second_time = 0, True, False
last_double_scan_time, last_single_scan_time, times_scanned = 0, 0, 0
prev_time = time.time()
lost_c1, lost_c2 = 51, 51

SERVO_INIT = {'x1': 65, 'y1': 90, 'x2': 115, 'y2': 90}
servo_angles = SERVO_INIT
vector_history = deque(maxlen=VECTOR_HISTORY_SIZE)
prev_position = None
balloon_position = triangulation(servo_angles)
balloon_positions_history = []
rotate_angles = {'x1': 0, 'y1': 0, 'x2': 0, 'y2': 0}

arduino = init_arduino(initial_angles=servo_angles)
cap1 = init_camera(CAMERA_INDEX_1)
cap2 = init_camera(CAMERA_INDEX_2)
frame1 = read_frame(cap1)
frame2 = read_frame(cap2)
x1_center, y1_center = calculate_center(frame1)
x2_center, y2_center = calculate_center(frame2)

pid_pan1, pid_tilt1 = init_pid(x1_center, y1_center)
pid_pan2, pid_tilt2 = init_pid(x2_center, y2_center)

print_div("Program started")
try:
    while True:
        print(f"Frame number: {frame_count}")
        current_time = time.time()
        frame_delay = current_time - prev_time
        total_time += frame_delay
        avg_frame_delay = total_time / frame_count if frame_count > 0 else frame_delay
        prev_time = current_time
        
        x1_correction, y1_correction = 0, 0
        detected_c1 = False
        frame1 = read_frame(cap1)
        results1 = MODEL(frame1, stream=True)
        for r1 in results1:
            if not r1.boxes:
                continue
            for box1 in r1.boxes:
                if valid_class(box1):
                    detected_c1 = True
                    draw_bounding_box(frame1, box1)
                    prev_position = draw_movement_vector(frame1, box1, balloon_position, vector_history, avg_frame_delay, prev_position)
                    x1_correction, y1_correction = calculate_corrections(box1, pid_pan1, pid_tilt1)
        
        x2_correction, y2_correction = 0, 0
        detected_c2 = False
        frame2 = read_frame(cap2)
        results2 = MODEL(frame2, stream=True)
        for r2 in results2:
            if not r2.boxes:
                continue
            for box2 in r2.boxes:
                if valid_class(box2):
                    detected_c2 = True
                    draw_bounding_box(frame2, box2)
                    prev_position = draw_movement_vector(frame2, box2, balloon_position, vector_history, avg_frame_delay, prev_position)
                    x2_correction, y2_correction = calculate_corrections(box2, pid_pan2, pid_tilt2)
        
        c1_change = x1_correction != 0 or y1_correction != 0
        c2_change = x2_correction != 0 or y2_correction != 0
        
        print(f"c1_change: {c1_change}, c2_change: {c2_change}, detected_c1: {detected_c1}, detected_c2: {detected_c2}, lost_c1: {lost_c1}, lost_c2: {lost_c2}, x1_correction: {x1_correction:.3f}, y1_correction: {y1_correction:.3f}, x2_correction: {x2_correction:.3f}, y2_correction: {y2_correction:.3f}")
        
        lost_c1 = 0 if detected_c1 else (lost_c1 + 1)
        lost_c2 = 0 if detected_c2 else (lost_c2 + 1)
        
        if c1_change and c2_change:
            balloon_position = triangulation(servo_angles)
            balloon_positions_history.append(balloon_position.copy())
            dist_c1 = get_distance('C1', balloon_position)
            dist_c2 = get_distance('C2', balloon_position)
            print(f"Servo angles: {servo_angles}")
            
            if first_time:
                last_prediction_time = current_time
                first_time, second_time = False, True
                
            elif valid_time(current_time, last_prediction_time, PREDICTION_INTERVAL) or (second_time and valid_time(current_time, last_prediction_time, FIRST_PREDICTION_INTERVAL)):
                lat, lon = calculate_lat_lon(balloon_position[0], balloon_position[1])
                #predictions = predict_balloon_flight(lat, lon)
                #plot_graph_trajectory(predictions)
                #plot_map_trajectory(predictions)
                # lyap_exp, lyap_time = compute_lyapunov_exponent(balloon_positions_history, avg_frame_delay)
                #send_telegram_alert(balloon_position, dist_c1, dist_c2)
                last_prediction_time, second_time = current_time, False
        
        if c1_change:
            y1_correction *= -1 # type: ignore
            servo_angles['x1'] = update_angle(servo_angles['x1'], x1_correction)
            servo_angles['y1'] = update_angle(servo_angles['y1'], y1_correction)
        
        if c2_change:
            y2_correction *= -1 # type: ignore
            servo_angles['x2'] = update_angle(servo_angles['x2'], x2_correction)
            servo_angles['y2'] = update_angle(servo_angles['y2'], y2_correction)
        
        if c1_change or c2_change:
            move_servos(arduino, servo_angles)
            last_double_scan_time, last_single_scan_time = current_time, current_time
        
        valid_double_scan_time = valid_time(current_time, last_double_scan_time, SERVO_DOUBLE_SCAN_INTERVAL)
        valid_single_scan_time = valid_time(current_time, last_single_scan_time, SERVO_SINGLE_SCAN_INTERVAL)
        
        if not c1_change and not c2_change:
            if ((times_scanned < (3 * SERVO_MAX / SERVO_SCAN_STEP)) or valid_double_scan_time) and (lost_c1 > 50 and lost_c2 > 50):
                if times_scanned >= (3 * SERVO_MAX / SERVO_SCAN_STEP):
                    times_scanned = 0
                rotate_angles = rotate_servos(arduino, servo_angles, times_scanned)
                last_double_scan_time = current_time
                times_scanned += 1
        elif not c1_change:
            if ((times_scanned < (3 * SERVO_MAX / SERVO_SCAN_STEP)) or valid_single_scan_time) and lost_c1 > 50:
                if times_scanned >= (3 * SERVO_MAX / SERVO_SCAN_STEP):
                    times_scanned = 0
                rotate_angles = rotate_servos(arduino, servo_angles, times_scanned, c2=False)
                last_single_scan_time = current_time
                times_scanned += 1
        elif not c2_change:
            if ((times_scanned < (3 * SERVO_MAX / SERVO_SCAN_STEP)) or valid_single_scan_time) and lost_c2 > 50:
                if times_scanned >= (3 * SERVO_MAX / SERVO_SCAN_STEP):
                    times_scanned = 0
                rotate_angles = rotate_servos(arduino, servo_angles, times_scanned, c1=False)
                last_single_scan_time = current_time
                times_scanned += 1
        
        frame_count += 1
        cv2.imshow("Image1", frame1)
        cv2.imshow("Image2", frame2)
        if cv2.waitKey(1) == ord('q'):
            break
finally:
    print_div("Program finished")
    #final_prints(frame_count, x1_center, y1_center, x2_center, y2_center, avg_frame_delay, 
                 #balloon_position, dist_c1, dist_c2, servo_angles, lyap_exp, lyap_time)
    cap1.release()
    cap2.release()
    cv2.destroyAllWindows()
    arduino.close()


---------------------------------------- Available cameras -----------------------------------------

1200: USB Camera VID:1133 PID:2085
1201: C270 HD WEBCAM
1202: iPhone Leo Camera
1203: MacBook Air Camera

----------------------------------------- Initial settings -----------------------------------------

Balloon position: (0.08, 0.16, 0.00)
Connected to Arduino on /dev/cu.usbmodem11301 at 9600 baud.
C1 → X: 65.0°, Y: 90.0°
C2 → X: 115.0°, Y: 90.0°
Camera 1200 opened successfully.
Camera 1201 opened successfully.
PID controller initialized.
PID controller initialized.

----------------------------------------- Program started ------------------------------------------

Frame number: 0

0: 480x640 1 person, 159.0ms
Speed: 1.8ms preprocess, 159.0ms inference, 0.1ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 147.9ms
Speed: 1.2ms preprocess, 147.9ms inference, 0.1ms postprocess per image at shape (1, 3, 480, 640)
c1_change: False, c2_change: False, det

KeyboardInterrupt: 

: 

In [2]:
enumerate_cameras()

NameError: name 'enumerate_cameras' is not defined